In [1]:
import pandas as pd

In [2]:
# Load Image Metadata
Image_Metadata = pd.read_csv("~/Git_Repos/UnsupervisedSegmentation/Metadata/Kidney_Annotations_Summary.csv")

# Determine path information
Image_Paths = Image_Metadata[["Path", "ManualClusterNumber"]].groupby("Path").max().reset_index()
Image_Paths["Path"] = "/Users/degn400/Git_Repos/UnsupervisedSegmentation/Images/Kidney_Tiles/Original/" + Image_Paths["Path"] + ".png"
Image_Paths["Path"] = [path.replace("_Annotations", "") for path in Image_Paths["Path"]]
Image_Paths

,Path,ManualClusterNumber
0,/Users/degn400/Git_Repos/UnsupervisedSegmentat...,5
1,/Users/degn400/Git_Repos/UnsupervisedSegmentat...,3
2,/Users/degn400/Git_Repos/UnsupervisedSegmentat...,4
3,/Users/degn400/Git_Repos/UnsupervisedSegmentat...,6
4,/Users/degn400/Git_Repos/UnsupervisedSegmentat...,5
5,/Users/degn400/Git_Repos/UnsupervisedSegmentat...,4
6,/Users/degn400/Git_Repos/UnsupervisedSegmentat...,5
7,/Users/degn400/Git_Repos/UnsupervisedSegmentat...,4
8,/Users/degn400/Git_Repos/UnsupervisedSegmentat...,3
9,/Users/degn400/Git_Repos/UnsupervisedSegmentat...,2


## Multi-Otsu

In [5]:
from PIL import Image
import skimage as ski
import numpy as np
import pandas as pd
import os

def run_multi_otsu(in_path, k, out_path):
    '''
    Apply multi_otsu threshold clustering to an image

    Args:
        in_path (string): Path to the input image 
        k (integer): Number of clusters
        out_path (string): Path to folder where outputs will be stored
    '''

    # Read the image 
    img = Image.open(in_path)

    # Convert the image to black and white
    img = img.convert("L")

    # Make the image a numpy array
    img_array = np.array(img)

    # Define thresholds     
    thresholds = ski.filters.threshold_multiotsu(img_array, classes = k)

    # Determine regions
    regions = np.digitize(img_array, bins = thresholds)

    # Since python starts counting from 0, add 1 to regions
    regions = regions + 1

    # Write image as pandas data.frame. Reverse image
    end_string = in_path.split("/")[-1].replace(".png", "_multiotsu.txt")
    pd.DataFrame(regions).iloc[::-1].reset_index(drop = True).to_csv(os.path.join(out_path, end_string), sep = "\t")
    return None

In [6]:
for row in range(len(Image_Paths)):
    run_multi_otsu(
        in_path = Image_Paths["Path"][row],
        k = Image_Paths["ManualClusterNumber"][row],
        out_path = "/Users/degn400/Git_Repos/UnsupervisedSegmentation/Images/Kidney_Tiles/Multiotsu_TXT/"
    )

## Binning

In [7]:
from PIL import Image
import numpy as np
import pandas as pd
import os

def run_binning(in_path, k, out_path):
    '''
    Apply bin threshold clustering to an image

    Args:
        in_path (string): Path to the input image 
        k (integer): Number of clusters
        out_path (string): Path to folder where outputs will be stored
    '''

    # Read the image 
    img = Image.open(in_path)

    # Convert the image to black and white
    img = img.convert("L")

    # Make a pandas dataframe of the image
    image_array = np.array(img)

    # Calculate thresholds
    minimum = np.min(image_array)
    maximum = np.max(image_array)
    diff = maximum - minimum
    multiplier = int(diff / k)
    thresholds = [multiplier * (x + 1) + minimum for x in range(k)]

    # Initialize a dataframe to hold clusters
    clusters = np.zeros(image_array.shape) + 1

    # Get indices where condition happens
    for thresh in thresholds[:-1]:
        index1, index2 = np.where(image_array > thresh)
        clusters[index1, index2] += 1

    # Write image as pandas data.frame. Reverse image
    end_string = in_path.split("/")[-1].replace(".png", "_binning.txt")
    pd.DataFrame(clusters.astype(int)).iloc[::-1].reset_index(drop = True).to_csv(os.path.join(out_path, end_string), sep = "\t")
    return None

In [8]:
for row in range(len(Image_Paths)):
    run_binning(
        in_path = Image_Paths["Path"][row],
        k = Image_Paths["ManualClusterNumber"][row],
        out_path = "/Users/degn400/Git_Repos/UnsupervisedSegmentation/Images/Kidney_Tiles/Binning_TXT/"
    )